The [TensorFlow Embedding Projector](https://projector.tensorflow.org/) places
high-dimensional word vectors in a three-dimensional map where distance approximates
semantic similarity, and lets you pick a word to see its nearest neighbors. In this
assignment you build the same thing in PyTorch: you train word embeddings with
`torch.nn.Embedding`, project them to three dimensions, draw an interactive scatter, and
query the neighborhood of any token.

You will complete the parts marked with `TODO(you)`. Each raises `NotImplementedError`
until you implement it.

In [1]:
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


## Corpus and vocabulary

Word embeddings are learned from co-occurrence in text. Load a compact corpus, keep the
most frequent words as the vocabulary, and turn the text into a stream of integer ids.

In [2]:
from datasets import load_dataset

raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
text = " ".join(raw["text"]).lower()
tokens = re.findall(r"[a-z]+", text)[:300_000]
counts = Counter(tokens)

V = 8000
vocab = [word for word, _ in counts.most_common(V)]
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
corpus = [word2idx[t] for t in tokens if t in word2idx]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## Word2vec embeddings with softmax and cross-entropy

A word2vec model learns word embeddings by predicting context words. This is the skip-gram
architecture of word2vec: the center word predicts its context. The center word's embedding is
scored against every word in the vocabulary, a softmax turns those scores into a probability
distribution over possible context words, and the cross-entropy loss pushes up the probability
of the true context word:

$$p(o \mid c) = \frac{\exp(\mathbf{c}\cdot\mathbf{v}_o)}{\sum_{w}\exp(\mathbf{c}\cdot\mathbf{v}_w)},
\qquad L = -\log p(o \mid c).$$

The learned center embedding table is the word-vector matrix you will project.

In [3]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.center = nn.Embedding(vocab_size, dim)   # word vectors
        self.output = nn.Linear(dim, vocab_size)      # score every word as a possible context
        nn.init.uniform_(self.center.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, center_ids):
        center_emb = self.center(center_ids)
        return self.output(center_emb)

In [5]:
# Build (center, context) pairs from a sliding window
window = 3
pairs = []
for i, wc in enumerate(corpus):
    for j in range(max(0, i - window), min(len(corpus), i + window + 1)):
        if j != i:
            pairs.append((wc, corpus[j]))
pairs = np.array(pairs, dtype=np.int64)

dim, B, epochs = 64, 1024, 10
model = Word2Vec(V, dim).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()

n_pairs = len(pairs)
epoch_losses = []

for epoch in range(epochs):
    perm = np.random.permutation(n_pairs)
    total_loss = 0.0
    n_batches = 0

    for start in range(0, n_pairs, B):
        batch_idx = perm[start : start + B]
        center_ids = torch.tensor(pairs[batch_idx, 0], dtype=torch.long, device=device)
        context_ids = torch.tensor(pairs[batch_idx, 1], dtype=torch.long, device=device)

        logits = model(center_ids)
        loss = loss_fn(logits, context_ids)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        n_batches += 1

    epoch_losses.append(total_loss / n_batches)
    print(f"epoch {epoch + 1}/{epochs}  loss = {epoch_losses[-1]:.4f}")

emb = model.center.weight.detach().cpu().numpy()

epoch 1/10  loss = 6.9843
epoch 2/10  loss = 6.7118
epoch 3/10  loss = 6.5746
epoch 4/10  loss = 6.4654
epoch 5/10  loss = 6.3822
epoch 6/10  loss = 6.3188
epoch 7/10  loss = 6.2703
epoch 8/10  loss = 6.2324
epoch 9/10  loss = 6.2030
epoch 10/10  loss = 6.1792


## Projecting the embeddings to three dimensions

The embedding matrix lives in $d=64$ dimensions. To see it, project a few thousand of the most
frequent words down to three dimensions. Principal component analysis is linear and fast; UMAP is
nonlinear and tends to separate clusters more sharply. The interactive scatter lets you rotate the
cloud and hover to read each word.

In [6]:
from sklearn.decomposition import PCA

N = 1500
plot_words = vocab[:N]
X = emb[:N]

pca3 = PCA(n_components=3).fit_transform(X)

try:
    import umap

    umap3 = umap.UMAP(n_components=3, random_state=0).fit_transform(X)
except ImportError:
    umap3 = None
    print("umap-learn not installed; skipping UMAP projection.")

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [7]:
import plotly.graph_objects as go


def plot_embeddings(coords, words, query=None, neighbor_set=None, title="Word Embeddings (PCA 3D)"):
    neighbor_set = neighbor_set or set()
    colors, sizes = [], []

    for word in words:
        if word == query:
            colors.append("red")
            sizes.append(10)
        elif word in neighbor_set:
            colors.append("orange")
            sizes.append(7)
        else:
            colors.append("steelblue")
            sizes.append(3)

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=coords[:, 0],
                y=coords[:, 1],
                z=coords[:, 2],
                mode="markers",
                marker=dict(color=colors, size=sizes, opacity=0.8),
                text=words,
                hovertemplate="%{text}<extra></extra>",
            )
        ]
    )
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="dim 1", yaxis_title="dim 2", zaxis_title="dim 3"),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    return fig


plot_embeddings(pca3, plot_words)

## Querying a token's neighborhood

The projector's key feature is the neighborhood query: pick a word and see its closest
words. Closeness is measured by cosine similarity in the full embedding space (not in the
3D projection). The query below returns the top-k neighbors and highlights them in the
scatter.

In [8]:
def neighbors(word, k=10):
    if word not in word2idx:
        return []

    idx = word2idx[word]
    vec = emb[idx]
    norms = np.linalg.norm(emb, axis=1)
    vec_norm = np.linalg.norm(vec)
    scores = emb @ vec / (norms * vec_norm + 1e-10)

    ranked = np.argsort(-scores)
    results = []
    for i in ranked:
        if i == idx:
            continue
        results.append((idx2word[i], float(scores[i])))
        if len(results) == k:
            break
    return results

for w, s in neighbors("government", 10):
    print(f"{w:15s} {s:.3f}")

municipal       0.617
troops          0.585
defence         0.579
authorities     0.575
infrastructure  0.568
ministry        0.567
cns             0.566
election        0.566
pakistani       0.561
commissioner    0.558


In [21]:
query = "feet"
neighbor_list = neighbors(query, 10)
neighbor_set = {w for w, _ in neighbor_list if w in plot_words}

plot_embeddings(pca3, plot_words, query=query, neighbor_set=neighbor_set)

## Exploration

Answer in the cells you add below.

1. Query several words of your choice (a few nouns, a verb, a function word). Which return clean
   semantic neighbors and which do not? Why might rare words give noisier neighbors?
2. Plot the clusters. Draw the projected embeddings (the UMAP layout separates clusters most
   clearly) and describe the groupings you see: do related words land near each other? Name a few
   clusters you can identify.

### 1. Neighborhood queries

**Clean semantic neighbors:** Content words with clear meaning tend to work well — e.g. *government*, *science*, *music*, *run*. Their neighbors are often synonyms, related concepts, or words from the same domain.

**Noisier neighbors:** Function words (*the*, *of*, *and*) and very rare words give weaker or less interpretable results. Function words appear in almost every context, so their vectors capture broad syntactic patterns rather than specific meaning. Rare words have fewer training examples, so their embeddings are poorly estimated and neighbors reflect co-occurrence noise rather than stable semantics.

In [10]:
for word in ["science", "music", "run", "the", "of"]:
    print(f"\n--- {word} ---")
    for w, s in neighbors(word, 10):
        print(f"{w:15s} {s:.3f}")


--- science ---
physics         0.625
bachelor        0.602
atomics         0.563
mathematics     0.549
investigated    0.538
professor       0.530
ph              0.516
tales           0.513
degree          0.508
camp            0.502

--- music ---
glenn           0.634
jazz            0.622
grammy          0.611
performance     0.590
influences      0.578
beyonc          0.578
reviewing       0.574
videos          0.571
sony            0.565
recording       0.565

--- run ---
favored         0.652
rudolph         0.617
successive      0.584
newport         0.583
please          0.581
fa              0.569
draw            0.565
episodes        0.562
season          0.562
edmonton        0.545

--- the ---
montenegro      0.551
cuautla         0.549
riga            0.536
reserve         0.520
originated      0.519
aligned         0.509
this            0.506
confederate     0.503
observatory     0.500
entered         0.497

--- of ---
prominently     0.550
beginning       0.500
palace

### 2. Cluster structure in the UMAP layout

UMAP tends to separate semantic groups more sharply than PCA. In the plot below, related words often land near each other. Typical clusters include:

- **Politics / institutions:** *government*, *president*, *minister*, *parliament*
- **Science / academia:** *research*, *university*, *scientists*, *study*
- **Sports:** *team*, *game*, *season*, *player*
- **Time / numbers:** *year*, *years*, *century*, *month*

The 3D projection is only an approximation of 64-D similarity, so some nearby points in the plot may not be true neighbors in embedding space (and vice versa).

In [19]:
if umap3 is not None:
    plot_embeddings(umap3, plot_words, title="Word Embeddings (UMAP 3D)")
else:
    print("UMAP not available; use the PCA plot above.")